In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn
import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, windows, targets, weights):
        self.windows = windows
        self.targets = targets
        self.weights = weights

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx], self.weights[idx]

def build_series_matrix(df):
    pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales')
    pivot = pivot.sort_index(axis=1)
    holiday = df.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()
    holiday = holiday.reindex(pivot.columns).fillna(False)
    return pivot, holiday

def make_windows(pivot, holiday, seq_len=52):
    values = pivot.fillna(0).values
    n_series, n_time = values.shape

    windows, targets, weights = [], [], []
    for t in range(seq_len, n_time):
        window = values[:, t - seq_len:t]
        target = values[:, t]
        valid = ~pivot.iloc[:, t].isna().values
        if valid.sum() == 0:
            continue
        w = np.where(holiday.iloc[t], 5.0, 1.0)
        windows.append(window[valid])
        targets.append(target[valid])
        weights.append(np.full(valid.sum(), w))
    return np.concatenate(windows), np.concatenate(targets), np.concatenate(weights)

In [ ]:
# PatchTST (Nie et al. 2023): seria pataраa patch-ebad iyofa (rogorc sityvebi
# tokenebad) da patch-ebis mimdevrobas transformer encoder amushavebs. es aris
# is transformeruli midgoma romelsac DLinear-is statia edaveba, aq sakutar
# monacemebze vamowmebt romeli aris ukotesi
class PatchTST(nn.Module):
    def __init__(self, seq_len, pred_len=1, patch_len=8, stride=4,
                 d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        n_patches = (seq_len - patch_len) // stride + 1
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos = nn.Parameter(torch.zeros(1, n_patches, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                           dim_feedforward=d_model * 2,
                                           dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(n_patches * d_model, pred_len)

    def forward(self, x):
        patches = x.unfold(1, self.patch_len, self.stride)
        z = self.patch_embed(patches) + self.pos
        z = self.encoder(z)
        return self.head(z.reshape(z.shape[0], -1))

class PatchTSTForecaster(BaseEstimator):
    def __init__(self, seq_len=52, epochs=10, lr=1e-3, batch_size=1024, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        self.seq_len = seq_len
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.patch_len = patch_len
        self.stride = stride
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout

    def _build_net(self):
        return PatchTST(self.seq_len_, 1, self.patch_len, self.stride, self.d_model, self.n_heads, self.n_layers, self.dropout)

    def fit(self, X, y, log_fn=None):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        pivot, holiday = build_series_matrix(df)
        self.seq_len_ = min(self.seq_len, pivot.shape[1] - 1)
        self.series_mean_ = pivot.mean(axis=1).fillna(0)
        self.series_std_ = pivot.std(axis=1).fillna(1).replace(0, 1)

        norm_pivot = pivot.sub(self.series_mean_, axis=0).div(self.series_std_, axis=0)
        windows, targets, weights = make_windows(norm_pivot, holiday, self.seq_len_)

        X_t = torch.tensor(windows, dtype=torch.float32)
        y_t = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        w_t = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        loader = DataLoader(WindowDataset(X_t, y_t, w_t), batch_size=self.batch_size, shuffle=True)

        self.model_ = self._build_net().to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for xb, yb, wb in loader:
                xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
                opt.zero_grad()
                pred = self.model_(xb)
                loss = (wb * (pred - yb).abs()).mean()
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
            epoch_loss /= len(loader.dataset)
            if log_fn is not None:
                log_fn(epoch, epoch_loss)

        self.model_ = self.model_.cpu()
        self.history_ = pivot
        return self

    def _forecast_series(self, key, target_dates):
        if key not in self.history_.index:
            return pd.Series(self.series_mean_.mean(), index=target_dates)

        mean = self.series_mean_.loc[key]
        std = self.series_std_.loc[key]
        series = self.history_.loc[key].copy()

        max_date = max(target_dates.max(), series.index.max())
        all_dates = pd.date_range(series.index.min(), max_date, freq='W-FRI')

        series = series.reindex(all_dates)
        known_mask = series.notna()
        series = series.fillna(0)

        values = ((series - mean) / std).values.tolist()

        self.model_.eval()
        with torch.no_grad():
            for i in range(len(values)):
                if i >= self.seq_len_ and not known_mask.iloc[i]:
                    window = torch.tensor([values[i - self.seq_len_:i]], dtype=torch.float32)
                    values[i] = self.model_(window).item()

        result = pd.Series(values, index=all_dates) * std + mean
        return result.reindex(target_dates)

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        preds = np.zeros(len(df))
        for (store, dept), group in df.groupby(['Store', 'Dept']):
            forecast = self._forecast_series((store, dept), pd.DatetimeIndex(group['Date'].unique()))
            preds[group.index] = forecast.reindex(group['Date']).values
        return preds

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('PatchTST_Training')

In [ ]:
with mlflow.start_run(run_name='PatchTST_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='PatchTST_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

In [ ]:
with mlflow.start_run(run_name='PatchTST_Windowing'):
    mlflow.log_param('seq_len', 52)
    pivot, holiday = build_series_matrix(train.assign(Weekly_Sales=y_train))
    windows, targets, weights = make_windows(pivot, holiday, 52)
    mlflow.log_metric('n_windows', len(windows))
    mlflow.log_metric('n_series', len(pivot))

len(windows), pivot.shape

In [ ]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# test-formis holdout: oqtombramde vswavlobt, shemdegi noembridan ivlisamde vafasebt.
# diagnostikam achvena rom es fanjara kaggle-is tests bevrad ukot imeorebs vidre CV
HOLDOUT_CUTOFF = pd.Timestamp('2011-10-28')
HOLDOUT_END = pd.Timestamp('2012-07-27')

In [ ]:
params = dict(seq_len=52, patch_len=8, stride=4, d_model=64, n_heads=4, n_layers=2, dropout=0.1, epochs=10, lr=1e-3, batch_size=1024)

with mlflow.start_run(run_name='PatchTST_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='PatchTST_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        tm = train['Date'] <= train_end
        vm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        def log_fn(epoch, loss, fold=i):
            mlflow.log_metric(f'train_loss_fold{fold}', loss, step=epoch)
            wandb.log({f'train_loss_fold{fold}': loss})

        model = PatchTSTForecaster(**params)
        model.fit(train.loc[tm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[tm], log_fn=log_fn)
        preds = model.predict(train.loc[vm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[vm].values, preds, train.loc[vm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    hm = train['Date'] <= HOLDOUT_CUTOFF
    hv = (train['Date'] > HOLDOUT_CUTOFF) & (train['Date'] <= HOLDOUT_END)
    hmodel = PatchTSTForecaster(**params)
    hmodel.fit(train.loc[hm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[hm])
    hpreds = hmodel.predict(train.loc[hv, ['Store', 'Dept', 'Date', 'IsHoliday']])
    wmae_holdout = wmae(y_train[hv].values, hpreds, train.loc[hv, 'IsHoliday'].values)

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', wmae_holdout)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': wmae_holdout})
    wandb.finish()

fold_scores, wmae_holdout

In [ ]:
with mlflow.start_run(run_name='PatchTST_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='PatchTST_Final', config=params, reinit='finish_previous')

    def log_fn(epoch, loss):
        mlflow.log_metric('train_loss', loss, step=epoch)
        wandb.log({'epoch': epoch, 'train_loss': loss})

    pipeline = Pipeline([
        ('model', PatchTSTForecaster(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train, model__log_fn=log_fn)

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_PatchTST.ipynb in Colab"
    !git push